In [ ]:
import requests, zipfile, io, os
import pandas as pd
import glob

RAW_DIR = "data/raw"
PROCESSED_DIR = "data/processed"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

S3_BASE = "https://s3.amazonaws.com/tripdata/"

def build_filename(year, month):
    """Handle the naming convention changes over time."""
    ym = f"{year}{month:02d}"
    if year < 2017:
        return f"{ym}-citibike-tripdata.zip"
    elif year < 2024:
        return f"{ym}-citibike-tripdata.csv.zip"
    else:
        return f"{ym}-citibike-tripdata.zip"

def download_month(year, month):
    """Download a single month's zip file."""
    fname = build_filename(year, month)
    local_path = os.path.join(RAW_DIR, fname)
    if os.path.exists(local_path):
        print(f"  Already exists: {fname}")
        return local_path

    url = S3_BASE + fname
    try:
        r = requests.get(url, stream=True, timeout=120)
        r.raise_for_status()
        with open(local_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"  Downloaded: {fname}")
        return local_path
    except requests.exceptions.HTTPError:
        # Try alternate naming
        alt = fname.replace(".zip", ".csv.zip") if ".csv.zip" not in fname else fname.replace(".csv.zip", ".zip")
        try:
            r = requests.get(S3_BASE + alt, stream=True, timeout=120)
            r.raise_for_status()
            local_path = os.path.join(RAW_DIR, alt)
            with open(local_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"  Downloaded (alt): {alt}")
            return local_path
        except Exception as e:
            print(f"  FAILED {year}-{month:02d}: {e}")
            return None

# Download every month from 2013-07 (system launch) through the latest available
for year in range(2013, 2026):
    for month in range(1, 13):
        if year == 2013 and month < 7:
            continue
        download_month(year, month)

: 

In [ ]:
def load_all_trips(raw_dir):
    """Unzip and concatenate all trip files into a single DataFrame."""
    frames = []
    for zpath in sorted(glob.glob(os.path.join(raw_dir, "*.zip"))):
        try:
            with zipfile.ZipFile(zpath) as z:
                for csv_name in z.namelist():
                    if not csv_name.endswith(".csv"):
                        continue
                    with z.open(csv_name) as f:
                        df = pd.read_csv(f, low_memory=False)
                        # Normalize column names to lowercase
                        df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
                        frames.append(df)
                        print(f"  Loaded {csv_name} from {os.path.basename(zpath)}: {len(df):,} rows")
        except Exception as e:
            print(f"  Error reading {zpath}: {e}")

    combined = pd.concat(frames, ignore_index=True)
    print(f"\nTotal rows: {len(combined):,}")
    print(f"Columns: {list(combined.columns)}")
    return combined

# This will take a while and use significant RAM.
# If RAM is limited, process month-by-month and save as parquet.
df = load_all_trips(RAW_DIR)
df.to_parquet(os.path.join(PROCESSED_DIR, "citibike_all.parquet"), index=False)

In [ ]:
def harmonize(df):
    """Normalize column names across different eras."""
    rename_map = {
        "tripduration": "duration_sec",
        "starttime": "started_at",
        "stoptime": "ended_at",
        "start_station_latitude": "start_lat",
        "start_station_longitude": "start_lng",
        "end_station_latitude": "end_lat",
        "end_station_longitude": "end_lng",
        "usertype": "member_casual",
        "bikeid": "rideable_id",
    }
    df = df.rename(columns=rename_map)

    # Map usertype values
    if "member_casual" in df.columns:
        df["member_casual"] = df["member_casual"].replace({
            "Subscriber": "member", "Customer": "casual"
        })

    # Compute duration if not present
    if "duration_sec" not in df.columns and "started_at" in df.columns and "ended_at" in df.columns:
        df["started_at"] = pd.to_datetime(df["started_at"])
        df["ended_at"] = pd.to_datetime(df["ended_at"])
        df["duration_sec"] = (df["ended_at"] - df["started_at"]).dt.total_seconds()

    return df